<a href="https://colab.research.google.com/github/berraEgcin/data_handling_project_26/blob/kerem-branch/notebooks/01_train_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.1 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/berraEgcin/data_handling_project_26.git
%cd data_handling_project_26
!git checkout kerem-branch

Cloning into 'data_handling_project_26'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 145 (delta 71), reused 116 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 205.36 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/data_handling_project_26
branch 'kerem-branch' set up to track 'origin/kerem-branch'.
Switched to a new branch 'kerem-branch'


In [6]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [4]:
import os
import torch
from google.colab import userdata
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# Load HF Token from Secrets if available
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    pass

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_DATA_PATH = "data/processed/train.jsonl"
VAL_DATA_PATH = "data/processed/val.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/clinical_lora_adapter"

def format_prompts(batch):
    formatted_texts = []
    for inp, out in zip(batch["input"], batch["output"]):
        text = (
            f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON.<|im_end|>\n"
            f"<|im_start|>user\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Loading data...")
dataset = load_dataset("json", data_files={"train": TRAIN_DATA_PATH, "validation": VAL_DATA_PATH})
dataset = dataset.map(format_prompts, batched=True)

# 1. 4-bit Quantization Config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. Tokenizer & Base Model
print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Force any non-quantized residual float parameters to float32 to prevent GradScaler conflicts
for name, module in model.named_modules():
    if "norm" in name or "lm_head" in name:
        module.to(torch.float32)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 4. SFTConfig: disable fp16 auto-scaling since QLoRA computes in 4-bit/fp16 under the hood
training_args = SFTConfig(
    output_dir="./tmp_checkpoints",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    fp16=False,
    bf16=False,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

# 5. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()

print(f"Saving adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training complete!")

Loading data...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/789 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.199587,0.210790,0.208166,161395.000000,0.915270
2,0.188876,0.203336,0.193964,322790.000000,0.918334
3,0.184881,0.197797,0.192178,484185.000000,0.918184


Saving adapter weights to /content/drive/MyDrive/clinical_lora_adapter...
Training complete!


In [7]:
import json
from transformers import pipeline

# Load a sample from the test set
with open("data/processed/test.jsonl", "r") as f:
    sample = json.loads(f.readline())

prompt = (
    f"<|im_start|>system\nYou are a clinical lab interpreter. Extract abnormal findings as valid JSON.<|im_end|>\n"
    f"<|im_start|>user\n{sample['input']}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

# Initialize text-generation pipeline with the fine-tuned model
generator = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
result = generator(prompt, max_new_tokens=256, do_sample=False)

raw_output = result[0]["generated_text"]
prediction = raw_output.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("=== PATIENT TEST INPUT ===")
print(sample["input"])
print("\n=== MODEL PREDICTED OUTPUT ===")
print(prediction)
print("\n=== EXPECTED GROUND TRUTH ===")
print(sample["output"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== PATIENT TEST INPUT ===
Patient Demographics: Gender=M. Lab Results: ALT: 58.3 U/L, Calcium: 8.8 mg/dL, Carbon Dioxide: 22.5 mmol/L, Chloride: 110.0 mmol/L, Glucose: 72.2 mg/dL, Potassium: 5.2 mmol/L, Sodium: 136.7 mmol/L, Urea Nitrogen: 7.8 mg/dL

=== MODEL PREDICTED OUTPUT ===
{"abnormal_findings": [{"parameter": "Carbon Dioxide", "value": 22.5, "unit": "mmol/L", "flag": "Low"}, {"parameter": "Potassium", "value": 5.2, "unit": "mmol/L", "flag": "High"}], "interpretation_summary": "Identified 2 abnormal parameter(s)."}

=== EXPECTED GROUND TRUTH ===
{"abnormal_findings": [{"parameter": "ALT", "value": 58.3, "unit": "U/L", "flag": "High"}, {"parameter": "Carbon Dioxide", "value": 22.5, "unit": "mmol/L", "flag": "Low"}, {"parameter": "Chloride", "value": 110.0, "unit": "mmol/L", "flag": "High"}, {"parameter": "Potassium", "value": 5.2, "unit": "mmol/L", "flag": "High"}], "interpretation_summary": "Identified 4 abnormal parameter(s)."}
